# Chapter 6 — An unbinned likelihood: the structure of Boötes I

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/06_likelihood.ipynb)

**Goal.** Once a satellite is found, measure its centre, size, ellipticity and orientation, and the number of its
stars, with a likelihood that uses every star individually — the approach of Martin, de Jong & Rix (2008,
https://arxiv.org/abs/0805.2945), who showed that it works with only tens of member stars. Then check, with
simulations, that the quoted uncertainties are honest, and see why the likelihood-ratio "significance" of a
detection must also be calibrated by simulation.

**You will learn**
1. to write the likelihood of a mixture of a satellite and a uniform background on the sky;
2. the elliptical exponential profile and its normalisation;
3. to estimate uncertainties from the curvature of the log-likelihood, and test their **coverage** with simulations;
4. the test statistic $TS = 2\Delta\ln\mathcal L$ and why $\sqrt{TS}$ is not a Gaussian significance here.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.optimize import minimize
from scipy import stats

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')
iso = load('isochrones_sdss')
ref = lvdb.loc['bootes_1']

## 6.1 The stars: a CMD mask at the distance of Boötes I

We select stars with the CMD mask of Chapter 5 at $\mu = 19.11$ (the catalogued distance; Chapter 2 measured it),
within 1° of the catalogued centre. Using a mask keeps the data a simple point set on the sky.

In [ ]:
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')
I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')
_c, _g = (I.g - I.r).to_numpy(), I.g.to_numpy()
_arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(_c), np.diff(_g)))])
_s = np.linspace(0, _arc[-1], 3000)
ISO_C, ISO_G, ISO_W = np.interp(_s, _arc, _c), np.interp(_s, _arc, _g), np.gradient(np.interp(_s, _arc, I.int_imf))


def satellite_cmd(mu, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    """Normalised satellite CMD density on the (CC, MM) grid for distance modulus mu (as in Chapters 4-5)."""
    pg = ISO_G + mu
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    S = np.einsum('k,kij->ij', ISO_W, np.exp(-0.5 * ((CC - ISO_C[:, None, None]) / sc[:, None, None])**2
                                             - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_mask(S, B, keep=0.5):
    """Cells ranked by S/B; the top ones that maximise sum(S)/sqrt(sum(B)) with sum(S) >= keep (Chapter 5)."""
    order = np.argsort((S / B).ravel())[::-1]
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])
    M = np.zeros(S.size, bool)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


cat = load('bootes1_sdss')
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
xi, eta = tangent(cat.ra.to_numpy(), cat.dec.to_numpy(), ref.ra, ref.dec)
H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
M = cmd_mask(satellite_cmd(ref.distance_modulus), B)
i, j = np.digitize(c0, cb) - 1, np.digitize(g0, mb) - 1
inmask = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm)) & M[np.clip(i, 0, len(cc) - 1), np.clip(j, 0, len(mm) - 1)]
RMAX = 1.0                                                        # degrees
use = inmask & (np.hypot(xi, eta) < RMAX)
x, y = xi[use] * 60, eta[use] * 60                                # arcmin
print(f'{use.sum()} stars in the CMD mask within {RMAX}°')

## 6.2 The model

**Satellite.** An elliptical exponential profile with centre $(x_0, y_0)$, half-light radius $r_h$ (along the major
axis), ellipticity $\epsilon = 1 - b/a$ and position angle $\theta$ of the major axis (from north through east). With
$u = (x-x_0)\sin\theta + (y-y_0)\cos\theta$ along the major axis and $v = (x-x_0)\cos\theta - (y-y_0)\sin\theta$
along the minor axis, the elliptical radius is $r_e = \sqrt{u^2 + v^2/(1-\epsilon)^2}$ and

$$ p_{\rm sat}(x, y) = \frac{1}{2\pi a^2 (1-\epsilon)}\, e^{-r_e/a}, \qquad a = r_h / 1.678, $$

normalised over the plane (the part beyond our 60′ circle is $< 10^{-4}$ for $r_h \approx 10′$).

**Background.** Uniform over the circle: $p_{\rm bg} = 1/(\pi R_{\max}^2)$.

**Likelihood.** With a satellite fraction $f$, for the $N$ stars,
$\ln\mathcal L = \sum_i \ln[f\, p_{\rm sat}(x_i, y_i) + (1-f)\, p_{\rm bg}]$. The number of satellite stars (in the mask)
is $N_\star = fN$.

One subtlety: if $r_h \to 0$ with the centre on one star, $p_{\rm sat}$ at that star diverges and so does $\mathcal L$.
A mixture likelihood is unbounded; we therefore impose $r_h \ge 0.5′$ (a "satellite" smaller than that is one star).

In [ ]:
R60 = RMAX * 60
P_BG = 1 / (np.pi * R60**2)
names = ['x0', 'y0', 'r_h', 'eps', 'theta', 'f']


def p_sat(x, y, x0, y0, rh, eps, theta):
    a = rh / 1.678
    t = np.radians(theta)
    u = (x - x0) * np.sin(t) + (y - y0) * np.cos(t)
    v = (x - x0) * np.cos(t) - (y - y0) * np.sin(t)
    return np.exp(-np.sqrt(u**2 + (v / (1 - eps))**2) / a) / (2 * np.pi * a**2 * (1 - eps))


RH_MIN = 0.5                                                     # arcmin: lower bound on r_h (see text)


def unpack(q):
    """Unconstrained vector -> parameters: r_h = RH_MIN + exp, eps and f via logistic functions; theta in [0, 180)."""
    return q[0], q[1], RH_MIN + np.exp(q[2]), 0.95 / (1 + np.exp(-q[3])), q[4] % 180, 1 / (1 + np.exp(-q[5]))


def nll(q, x, y):
    x0, y0, rh, eps, th, f = unpack(q)
    return -np.sum(np.log(f * p_sat(x, y, x0, y0, rh, eps, th) + (1 - f) * P_BG))


def fit(x, y, start=(0, 0, np.log(8), 0.0, 30.0, -1.0)):
    """Maximum-likelihood fit; several starting position angles guard against local minima."""
    best = None
    for th0 in (0, 60, 120):
        q0 = np.array(start, float); q0[4] = th0
        r = minimize(nll, q0, args=(x, y), method='Nelder-Mead', options=dict(maxiter=6000, xatol=1e-5, fatol=1e-6))
        if best is None or r.fun < best.fun:
            best = r
    return best


def hessian_errors(q, x, y):
    """1-sigma errors of (x0, y0, r_h, eps, theta, f) from the numerical Hessian of -ln L in physical parameters."""
    p = np.array(unpack(q))
    to_q = lambda p: np.array([p[0], p[1], np.log(p[2] - RH_MIN), -np.log(0.95 / p[3] - 1), p[4], -np.log(1 / p[5] - 1)])
    f = lambda p: nll(to_q(p), x, y)
    h = np.array([0.05, 0.05, 0.02 * p[2], 0.01, 1.0, 0.002])
    Hm = np.zeros((6, 6))
    for a in range(6):
        for b in range(6):
            ea, eb = np.eye(6)[a] * h[a], np.eye(6)[b] * h[b]
            Hm[a, b] = (f(p + ea + eb) - f(p + ea - eb) - f(p - ea + eb) + f(p - ea - eb)) / (4 * h[a] * h[b])
    return p, np.sqrt(np.diag(np.linalg.inv(Hm)))


best = fit(x, y)
p, err = hessian_errors(best.x, x, y)
lnl0 = len(x) * np.log(P_BG)                                       # f = 0: background only
TS = 2 * (-best.fun - lnl0)
for n, v, e in zip(names, p, err):
    print(f'{n:6s} = {v:8.3f} ± {e:.3f}')
print(f'N_star = {p[5] * len(x):.0f}; TS = {TS:.0f}')
print(f'LVDB v1.1.1:          r_h = {ref.rhalf:.2f}′, eps = {ref.ellipticity:.2f}, PA = {ref.position_angle:.0f}°')
# Martin, de Jong & Rix (2008, Table 1; https://arxiv.org/abs/0805.2945): the same SDSS data, a similar likelihood
M08 = dict(rh=12.6, rh_err=1.0, eps=0.39, eps_err=0.06, pa=14, pa_err=6)
print(f'Martin et al. (2008): r_h = {M08["rh"]}′ ± {M08["rh_err"]}, eps = {M08["eps"]} ± {M08["eps_err"]}, PA = {M08["pa"]} ± {M08["pa_err"]}°')

Our centre, ellipticity and position angle agree with both references. The half-light radius agrees with Martin et
al. (2008), who fitted the **same SDSS data** with a similar likelihood, but is ~30% (2.5σ) larger than the LVDB
value, taken from the deeper imaging of Muñoz et al. (2018, https://ui.adsabs.harvard.edu/abs/2018ApJ...860...66M; the LVDB
`ref_structure` column). The LVDB documentation does not say whether `rhalf` is a major-axis or circularised radius;
we assume major-axis (UNVERIFIED). A structural parameter is a property
of a *data set and a model*, not only of the object: depth, the CMD selection, the profile family and the treatment of
the outskirts all matter. We therefore test against the same-data measurement, and keep the LVDB comparison as a warning.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(x, y, s=2, c='k')
t = np.linspace(0, 2 * np.pi, 200)
x0, y0, rh, eps, th, f = p
for k in (1, 2, 3):
    u, v = k * rh * np.cos(t), k * rh * (1 - eps) * np.sin(t)
    tr = np.radians(th)
    ax[0].plot(x0 + u * np.sin(tr) + v * np.cos(tr), y0 + u * np.cos(tr) - v * np.sin(tr), c='C3')
ax[0].set(xlim=(30, -30), ylim=(-30, 30), aspect='equal', xlabel="ξ [arcmin]", ylabel="η [arcmin]",
          title='stars in the CMD mask; ellipses at 1, 2, 3 $r_h$')
# radial profile in elliptical annuli, with the model
tr = np.radians(th)
u = (x - x0) * np.sin(tr) + (y - y0) * np.cos(tr)
v = (x - x0) * np.cos(tr) - (y - y0) * np.sin(tr)
re = np.sqrt(u**2 + (v / (1 - eps))**2)
edges = np.arange(0, 45.1, 3)
n = np.histogram(re, edges)[0]
area = np.pi * (1 - eps) * np.diff(edges**2)
rr = 0.5 * (edges[1:] + edges[:-1])
ax[1].errorbar(rr, n / area, np.sqrt(n) / area, fmt='o', c='k', label='data')
ax[1].plot(rr, f * len(x) * p_sat(rr * np.sin(tr) + x0, rr * np.cos(tr) + y0, x0, y0, rh, eps, th) + (1 - f) * len(x) * P_BG, c='C3',
           label='model')
ax[1].axhline((1 - f) * len(x) * P_BG, c='0.5', ls=':', label='background')
ax[1].set(yscale='log', xlabel='elliptical radius [arcmin]', ylabel='stars per arcmin²')
ax[1].legend()
plt.tight_layout()

dpa = abs((p[4] - ref.position_angle + 90) % 180 - 90)
assert np.hypot(p[0], p[1]) < 2                                   # centre within 2' of LVDB
assert abs(p[2] - M08['rh']) < 2 * np.hypot(err[2], M08['rh_err'])  # r_h within 2 combined sigma of Martin+08
assert abs(p[3] - ref.ellipticity) < 0.15 and abs(p[3] - M08['eps']) < 0.15
assert dpa < 15

**Figure 6.1.** Left: the stars of the CMD mask, with the best-fitting ellipses. Right: star density in elliptical
annuli and the model. The exponential profile describes Boötes I until it sinks into the background at ~30′
(≈ 2.5 $r_h$).

## 6.3 Are the error bars honest? A coverage test

The errors above come from the curvature of $\ln\mathcal L$ at its maximum, which is exact only for a Gaussian
likelihood. We test them: simulate 200 data sets from the best-fit model (same total N, satellite stars drawn from
the profile, background uniform), fit each, and count how often the true value lies within the quoted 1σ error.
For honest errors this happens in 68% of the simulations (with 200 simulations, the binomial scatter is ±3.3%).

In [ ]:
rng = np.random.default_rng(6)


def simulate(p, n):
    """Positions of n stars drawn from the model with parameters p (arcmin)."""
    x0, y0, rh, eps, th, f = p
    ns = rng.binomial(n, f)
    a = rh / 1.678
    r = rng.gamma(2, a, ns)                                       # exponential surface density: r ~ Gamma(2, a)
    phi = rng.uniform(0, 2 * np.pi, ns)
    u, v = r * np.cos(phi), r * np.sin(phi) * (1 - eps)
    tr = np.radians(th)
    xs, ys = x0 + u * np.sin(tr) + v * np.cos(tr), y0 + u * np.cos(tr) - v * np.sin(tr)
    keep = np.hypot(xs, ys) < R60
    rb, pb = R60 * np.sqrt(rng.uniform(0, 1, n - keep.sum())), rng.uniform(0, 2 * np.pi, n - keep.sum())
    return np.concatenate([xs[keep], rb * np.cos(pb)]), np.concatenate([ys[keep], rb * np.sin(pb)])


cover = []
for k in range(200):
    xs, ys = simulate(p, len(x))
    b = fit(xs, ys, start=(0, 0, np.log(p[2] - RH_MIN), 0.0, p[4], best.x[5]))
    ps, es = hessian_errors(b.x, xs, ys)
    d = ps - p
    d[4] = (d[4] + 90) % 180 - 90                                 # position angle is defined modulo 180
    cover.append(np.abs(d) < es)
cover = np.array(cover).mean(0)
for n_, c in zip(names, cover):
    print(f'{n_:6s}: true value within 1 sigma in {c:.2f} of simulations')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(names, cover)
ax.axhspan(0.68 - 2 * 0.033, 0.68 + 2 * 0.033, color='C1', alpha=0.3, label='0.68 ± 2σ (binomial, 200 sims)')
ax.set(ylabel='coverage of the 1σ interval', ylim=(0, 1))
ax.legend()
plt.tight_layout()
assert 0.60 < cover[2] < 0.76                                     # the half-light radius

**Figure 6.2.** Coverage of the 1σ Hessian errors. With ~400 satellite stars the likelihood is close to Gaussian:
the errors of $r_h$, ε and θ are honest (coverage 0.64–0.67), while those of the satellite fraction and one centre
coordinate are slightly optimistic (~0.60, about 2.5 binomial σ below 0.68). For a satellite with tens of stars the
Hessian would be far worse: then use the profile likelihood or a posterior sample (e.g. MCMC).

## 6.4 How significant is a detection? Why not $\sqrt{TS}$

The test statistic compares the best satellite model with the background-only model, $TS = 2[\ln\mathcal L_{\max} -
\ln\mathcal L_0]$. For Boötes I, TS ≈ 390: no doubt. For a marginal candidate one is tempted to
quote $\sqrt{TS}$ as "the number of sigmas" (Wilks' theorem: TS ~ $\chi^2_k$ under the null). Wilks' theorem does
not apply here: $f \ge 0$ sits on the boundary of the parameter space at the null, and the centre, size, ellipticity
and angle are *undefined* when $f = 0$. Fitting pure-background simulations shows what TS looks like with no
satellite.

In [ ]:
ts_null = []
for k in range(100):
    xs, ys = simulate((0, 0, 8, 0.3, 0, 0.0), len(x))             # f = 0: background only
    b = min((fit(xs, ys, start=(sx, sy, np.log(4), 0, 0, -3)) for sx, sy in rng.uniform(-40, 40, (3, 2))), key=lambda r: r.fun)
    ts_null.append(2 * (-b.fun - len(xs) * np.log(P_BG)))
ts_null = np.array(ts_null)
print(f'null TS: median {np.median(ts_null):.1f}, 95th percentile {np.percentile(ts_null, 95):.1f}; '
      f'chi2 with 1 dof would give 95th percentile {stats.chi2.ppf(0.95, 1):.1f}')
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(ts_null, 20, density=True, label='fits to pure-background simulations')
tt = np.linspace(0.01, ts_null.max(), 200)
ax.plot(tt, stats.chi2.pdf(tt, 1), label=r'$\chi^2_1$')
ax.set(xlabel='TS', ylabel='density')
ax.legend()
plt.tight_layout()

**Figure 6.3.** TS of fits to background-only simulations, with only a few starting positions per fit (a real search
would scan the whole field, making the null TS larger still). Its distribution is far wider than $\chi^2_1$: a TS of 10
is not "3σ". As in Chapter 5, significance must be calibrated by simulating the whole procedure.

## Summary

- A mixture of an elliptical exponential and a uniform background, fitted star by star, measures the structure of
  Boötes I; it reproduces the SDSS measurement of Martin et al. (2008). Deeper data give a smaller $r_h$: compare
  measurements made with matching data and definitions.
- Hessian errors are close to honest here (coverage 0.59–0.67) because the satellite has hundreds of stars; check
  this, don't assume it.
- $TS$ does not follow $\chi^2$ for detections; calibrate with simulations.